# 04a · Single-LLM baseline  _(owner: M4)_

One prompt, no agents: explain a clinical note to a patient in plain language. Run on 3 MTSamples notes and 10 PLABA abstracts, saving each output + time. Then the same PLABA run on a second model (gemma2:2b) to compare speed.

In [1]:
import json
import time
import pandas as pd
import requests
from IPython.display import Markdown, display

df = pd.read_csv("../data/mtsamples.csv")
plaba = json.load(open("../data/plaba/data.json"))

MODEL = "qwen3.5:2b"

In [2]:
# Call Ollama directly with thinking off. Otherwise qwen3.5 thinks until it times out and returns nothing.
def call_model(system, user, model=MODEL):
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            "think": False,
            "stream": False,
        },
        timeout=600,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]

In [3]:
# Baseline: one instruction, no agents. Used for every run below.
system = "You are a helpful medical assistant who explains clinical notes to patients."
prompt = "Explain this clinical note to the patient in plain language.\n\n"

## MTSamples (3 notes)

In [4]:
# 3 MTSamples notes by row ID: discharge summary, clinic visit, surgery report.
# data/ is git-ignored, so no note text goes in the repo.
NOTE_IDS = [3889, 1295, 184]

notes = [
    {"id": i, "name": df.loc[i, "sample_name"].strip(), "text": df.loc[i, "transcription"]}
    for i in NOTE_IDS
]

for note in notes:
    print(note["id"], note["name"], "-", len(note["text"].split()), "words")

3889 Pyelonephritis - Discharge Summary - 174 words
1295 URI - SOAP - 247 words
184 Wrist Ganglion Excision - 156 words


In [5]:
# Run the baseline on each note and time it.
results = []

for note in notes:
    start = time.time()
    output = call_model(system, prompt + note["text"], model=MODEL)
    seconds = round(time.time() - start, 2)

    results.append({"note_id": note["id"], "name": note["name"], "seconds": seconds, "output": output})
    print(note["id"], note["name"], "took", seconds, "s")

3889 Pyelonephritis - Discharge Summary took 22.47 s


1295 URI - SOAP took 19.74 s


184 Wrist Ganglion Excision took 10.98 s


In [6]:
# Time for each output.
pd.DataFrame(results)[["note_id", "name", "seconds"]]

,note_id,name,seconds
0,3889,Pyelonephritis - Discharge Summary,22.47
1,1295,URI - SOAP,19.74
2,184,Wrist Ganglion Excision,10.98


In [7]:
# Show each output as formatted text (VS Code cuts off long plain-text output).
for r in results:
    display(Markdown(f"## {r['note_id']} {r['name']} ({r['seconds']} s)\n\n{r['output']}"))

## 3889 Pyelonephritis - Discharge Summary (22.47 s)

Here is a plain-language explanation of your medical notes, broken down by topic:

### **1. What is wrong with you right now?**
Your doctor has diagnosed you with a few conditions that happened together. Here are the simple terms for each diagnosis and what it likely means:
*   **Pyelonephritis:** This is an infection in your kidneys. It started because bacteria (specifically *Klebsiella*) were growing in the urine. Since there was a tube coming from your bladder to a specific part of your intestines called an "ileal conduit," it seems like there may have been some thick build-up of mucus inside that tube (in the Foley), which let this bacteria enter and cause the infection.
*   **Hypertension:** Your blood pressure is currently high. You likely had this before or recently, but you do not need to worry about it right now.
*   **Anemia:** This is a condition where your body doesn't have enough healthy red blood cells. You've mentioned this has been happening for over a year, so it is chronic. While the other things I'm treating now might help boost your energy and reduce blood loss, this takes a long time to fix.

### **2. What happened during your hospital stay?**
*   **The Infection:** We treated the kidney infection with IV antibiotics because of how hard the bacteria were hiding.
*   **The Surgery:** After you improved (got better), doctors performed surgery on your bowel to clean out the thick mucus plug in your urine tube. This was done because it was blocking your urethra.
*   **The Antibiotics:** You were treated with pills containing **Bactrim** (which is a combination of sulfamethoxazole and trimethoprim). The good news is:
    *   Your symptoms got better quickly after the antibiotics.
    *   Your kidneys worked fine, and you are no longer sick with the infection.
*   **The Plan:** You were scheduled to have another surgery in about 30 days to check your tubes again or clean them up further.
*   **Follow-up:** Make sure you call Dr. Y at home in ten days so they can see if everything is still healthy.

### **3. The other symptoms you mentioned**
You also had a knee injury complaint while waiting for the surgery:
*   **The Injury:** It sounds like you had sudden, sharp pain on one side of your right leg (the patellar tendon area) with no swelling in the joint itself.
*   **The Diagnosis:** Your doctors did not believe it was an "intraarticular" injury (which would mean inside the knee joint). Based on the pattern of the pain and lack of swelling, they think this is just tenderness or perhaps a minor muscle strain, rather than a serious break or torn ligament.
*   **The Advice:** Because the doctors were not sure, you do not have to take an ambulance ride home immediately. Instead, they told you it was okay for you to take over-the-counter **Ibuprofen** (about 2 or 3 tablets twice a day) to help reduce pain and swelling.

### **Summary Checklist for Next Steps:**
1.  **Antibiotics:** Continue taking your Bactrim daily until the infection is fully resolved. Do not stop early.
2.  **Pain Medicine:** Take the Ibuprofen as advised to manage the knee pain you felt.
3.  **Surgery Appointment:** Go to Dr. Y in 10 days for the tube inspection.
4.  **Knee Care:** Monitor your knee pain and swelling. If it gets worse or if you feel a sudden sharp "pop" or give up going on a trip, tell Dr. Y immediately at home before the appointment, as they will likely take you to the hospital in that case.
5.  **Blood Count:** Since you have anemia, monitor your hemoglobin levels closely. Ask your doctor if you should be getting blood transfusions anytime soon.

*Disclaimer: I am an AI assistant, not a doctor. This explanation is for educational purposes only and does not replace the advice of your treating physicians.*

## 1295 URI - SOAP (19.74 s)

Hi! I'd be happy to explain what these clinical notes mean for your mom in simple terms. This note is a summary of her visit today from a doctor, written in the jargon used by health professionals. Let's break it down line by line:

**The Main Issue:**
Your mom brought her 13-year-old son in because he has a sore throat that started overnight. The biggest physical signs are that his eyes are very puffy (swollen), he feels very run down or tired, and his voice sounds hoarse. He wasn't coughing, but there was some discharge seen behind his throat.

**Why he feels sick:**
*   **No Bacteria:** The doctor tested this, and the Strep test came back negative. This is important because it means a severe "strep throat" or ear infection caused by bacteria like *Streptococcus pyogenes* is **not** present. It also rules out sinusitis (sinus infection).
*   **No Cough:** The note specifically says he has no cough, which helps rule out conditions like a common cold with severe congestion or other infectious causes that usually present with it.

**What likely happened:**
It sounds like he may have caught something viral (like the flu, rhinovirus, or some cold virus) recently, or perhaps developed a mild allergic reaction to something just yesterday. The sudden onset of puffy eyes and a sore throat without coughing is often associated with upper respiratory infections that start with congestion but don't progress fully into other symptoms. His "runny" voice could be due to inflammation (which happens when there is fluid) or dehydration, though he is still eating and drinking well.

**What was done:**
*   **Tests:** The doctor performed a throat swab for bacteria (negative) and checked his sinuses with an X-ray (also negative). These tests confirmed there are no serious bacterial infections present right now.
*   **Treatment Plan:** Because the tests didn't show a serious bacterial infection, the doctor decided **not** to give antibiotics. Instead, he is treating it as a "minor illness" or viral congestion.
    *   He prescribed **Levall**, which appears to be an over-the-counter medication designed to clear nasal congestion.
    *   He also mentioned using an expectorant (a cough suppressant) if needed, though his note notes he has no coughing.
    *   He is encouraged to drink plenty of fluids and rest.

**How you can help:**
*   **Stay hydrated.** Since your son feels "run down," drinking water will help him feel better.
*   **Medication:** As the note suggests, let's see if he responds to the congestion medication (Levall). If he gets a fever back or becomes more uncomfortable, please call him again for Tylenol (Acetaminophen) or Ibuprofen (as an alternative to Levall, though Tylenol is often preferred for kids/young adults with no stomach issues).
*   **Rest:** He needs plenty of rest.

**When to worry (Red Flags):**
Even though the tests are normal and he feels okay right now, you should be aware that "run down" and a sore throat can sometimes signal something serious if accompanied by:
*   Difficulty breathing or breathing in short gasps.
*   A severe, persistent fever that doesn't go down.
*   Swelling of the tongue or soft palate (the roof of the mouth).
*   Lethargy (extreme tiredness that makes walking hard to the point of fainting).

**Bottom Line:**
Your son seems to have caught a mild viral infection causing sore throat and congestion. He has not had any bacteria or severe sinus infections, so he is likely safe to go home with observation for now. However, please monitor him closely. If he gets worse, feels much more confused or extremely lethargic, or develops a new fever that doesn't come down, do call the doctor again.

*Disclaimer: I am an AI assistant providing information based on this text. This explanation is not a substitute for professional medical advice, diagnosis, or treatment.*

## 184 Wrist Ganglion Excision (10.98 s)

Good afternoon! I'm here to help you understand what happened during your surgery. You can ask me anything if you have more questions later.

Here is a simple explanation of your preoperative and postoperative notes:

**Before Your Surgery**
First, we put you under **general anesthesia**, which means you likely didn't even remember the experience while having your arm numbed. Before starting, you were given **antibiotics** to help prevent infection. Because of the number of nerves nearby in that area, it's common for patients to feel some numbness or tingling after surgery; this is completely normal and will go away on its own.

**What We Did**
We focused on your right wrist (the side with your hand). Specifically, we had a **ganglion**, which is a small fluid-filled lump that often forms under the skin of the wrist due to arthritis or irritation from wearing shoes for long periods.

The procedure involved making an incision directly over the lump to remove it along with its stalk. Additionally, doctors removed a small piece of skin (about 1 square centimeter) around where the ganglion was attached to prevent this lump from forming a "one-way valve," which would let fluid get trapped behind it and grow larger. This step is very important to make sure your wrist stays healthy long-term.

**After the Surgery**
We placed a special compression wrap (called an Esmarch) on your arm and put you in a tourniquet—a tight band around your arm for about 30 minutes—to stop blood flow temporarily so we could see everything clearly. After the surgery, we washed the area with soap and water, covered it with a sterile dressing, applied splints (bandages) to protect the area, and injected pain relief medicine called **Marcaine with epinephrine**.

You were sent straight to your recovery room feeling well. We took a few moments to check that everything was done correctly and that you weren't experiencing any issues. You are feeling very much better now!

***
**In short:** Your wrist ganglion is removed, the surrounding skin is slightly resected to keep it from growing back, pain relief is given, and your arm is bandaged while you recover. You were doing great throughout the process. Please feel free to ask me anything else about your care after this visit!

### Notes

- 11–22 s per note. Outputs are long (380–640 words), 2–3x longer than the notes themselves.

Mistakes I found by reading each output against its note

3889 Pyelonephritis discharge
- Says ibuprofen is "2 or 3 tablets twice a day", but the note says t.i.d., which means three times a day.
- Says doctors did "surgery on your bowel" to clear the mucus plug, and that it was "blocking your urethra". The catheter was flushed (irrigated), not operated on, and the plug was in the catheter in the ileal conduit.
- Explains the ileal conduit as "a tube coming from your bladder to... your intestines", but it's a urine pathway made from a piece of intestine.
- Says "your kidneys worked fine", but the note lists mild renal insufficiency (kidneys working below normal).
- Mixes up the follow-up and the surgery: "Go to Dr. Y in 10 days for the tube inspection", and the surgery is "in about 30 days". The note gives a set surgery date and a separate follow-up in 7–10 days.
- Adds that blood pressure is nothing "to worry about right now", to "monitor your hemoglobin", to ask about "blood transfusions", and to take Bactrim until the infection is gone.
- Garbles the knee advice ("you do not have to take an ambulance ride home", "if you... give up going on a trip, tell Dr. Y").

1295 URI SOAP
- Talks to the wrong person: "explain what these clinical notes mean for your mom". The patient is the 13-year-old; his mom brought him in.
- Calls the expectorant "a cough suppressant", but an expectorant loosens mucus, which is close to the opposite.
- Says Levall "appears to be an over-the-counter medication" that was "prescribed", but the doctor gave samples of it.
- Says having no cough "helps rule out... a common cold", but the diagnosis is an upper respiratory infection (a cold).
- Offers Tylenol or ibuprofen "as an alternative to Levall", but the note says they're for discomfort, alongside Levall.
- Adds that it may be an allergic reaction or the flu, and warning signs (tongue swelling, fever, extreme tiredness).
- Leaves out the Benadryl he already took, and the history of high blood pressure and weight gain.

184 Wrist ganglion excision
- Says they removed "a small piece of skin", but it was a piece of the joint capsule (the lining around the joint).
- Puts the Esmarch and tourniquet "after the surgery", but they were used before the cut to stop blood flow.
- Says ganglions come from "wearing shoes for long periods", which is made up.
- Adds "your right wrist" (the note doesn't say which side), that the tourniquet was on "for about 30 minutes", that numbness and tingling are normal, and "you are feeling very much better now".
- Uses confusing phrases like "your right wrist (the side with your hand)" and "we had a ganglion".

Patterns

Wrong explanations of jargon are still the biggest problem: abbreviations (t.i.d.), anatomy (ileal conduit, capsule), and drug roles (expectorant), and the model sounds confident even when it's wrong. Adding things not in the note happened in all 3, including made-up timings (30 days, 30 minutes), warning signs, and medical advice (transfusions). The model also mixes up who and when: it talks to the mom instead of the patient, puts the tourniquet after surgery, and merges the follow-up with the surgery. Most fixes are small (one word, one phrase, or deleting an invented sentence), not full rewrites, which fits what the Refiner is meant to do.

## PLABA (10 abstracts)

In [8]:
# 10 PLABA pairs: for questions 1-10, the first abstract with 100-250 words.
# Each pair = the abstract + a human-written plain-language version (the reference).
# PLABA can't be redistributed, so we never print the abstract or reference text.
examples = []

for q in [str(i) for i in range(1, 11)]:
    for pmid, a in plaba[q].items():
        if not isinstance(a, dict):
            continue
        abstract = " ".join(a["abstract"].values())
        if 100 <= len(abstract.split()) <= 250:
            ref_key = sorted(a["adaptations"])[0]
            reference = " ".join(a["adaptations"][ref_key].values())
            examples.append({"id": pmid, "question_id": q, "title": a["Title"], "text": abstract, "reference": reference})
            break

for ex in examples:
    print(ex["question_id"], ex["id"], ex["title"][:60], "-", len(ex["text"].split()), "words")

1 15902691 Muscle cramps - 122 words
2 30838456 Serotonin and Norepinephrine Reuptake Inhibitors - 177 words
3 16445274 Hyperkalemia - 171 words
4 9803200 Diagnosis and classification of diabetes mellitus: new crite - 188 words
5 21636864 Arthroscopic treatment of popliteal cyst - 138 words
6 16127951 Hyperthyroidism: diagnosis and treatment - 182 words
7 15156437 Meta-analysis of cephalosporins versus penicillin for treatm - 161 words
8 1000139 Clozapine versus perphenazine: the value of the biochemical  - 165 words
9 7698623 Management of foreign bodies of the upper gastrointestinal t - 217 words
10 3068693 Finger pain - 217 words


In [9]:
# Run the baseline on each abstract and time it.
plaba_results = []

for ex in examples:
    start = time.time()
    output = call_model(system, prompt + ex["text"], model=MODEL)
    seconds = round(time.time() - start, 2)

    plaba_results.append({"id": ex["id"], "title": ex["title"], "seconds": seconds, "output": output})
    print(ex["id"], "took", seconds, "s")

15902691 took 11.77 s


30838456 took 15.01 s


16445274 took 12.92 s


9803200 took 14.47 s


21636864 took 12.27 s


16127951 took 18.4 s


15156437 took 13.35 s


1000139 took 9.91 s


7698623 took 12.92 s


3068693 took 18.2 s


In [10]:
# Time for each output.
pd.DataFrame(plaba_results)[["id", "title", "seconds"]]

,id,title,seconds
0,15902691,Muscle cramps,11.77
1,30838456,Serotonin and Norepinephrine Reuptake Inhibitors,15.01
2,16445274,Hyperkalemia,12.92
3,9803200,Diagnosis and classification of diabetes melli...,14.47
4,21636864,Arthroscopic treatment of popliteal cyst,12.27
5,16127951,Hyperthyroidism: diagnosis and treatment,18.40
6,15156437,Meta-analysis of cephalosporins versus penicil...,13.35
7,1000139,Clozapine versus perphenazine: the value of th...,9.91
8,7698623,Management of foreign bodies of the upper gast...,12.92
9,3068693,Finger pain,18.20


In [11]:
# Show each output as formatted text (VS Code cuts off long plain-text output).
for r in plaba_results:
    display(Markdown(f"## {r['id']} {r['title']} ({r['seconds']} s)\n\n{r['output']}"))

## 15902691 Muscle cramps (11.77 s)

Hi there! I'd be happy to explain what your clinical note means, written in plain English so it's easy for you to understand.

Here is a simplified breakdown of the situation:

**What is happening?**
Your muscle (muscles) have suddenly caught up and squeezed tight on their own. This makes you feel very uncomfortable or even painful. It's not just normal muscle "twitching" that feels this way; these are deep, involuntary cramps triggered by nerve signals.

**Why does it happen?**
Doctors don't know exactly why some of you get these cramps as often as others do (though the note says they are likely caused by factors in your medical history, how much exercise you do, or blood tests). To figure out what's causing your specific case, doctors will look at your past health, listen to what sounds when you move, and check a few simple labs.

**Why is it an issue?**
Even though the note calls these "benign" (meaning harmless in a basic sense), living with them is very unpleasant for many people. That's why doctors treat them carefully. Currently, there aren't enough studies to say exactly how they work at a cellular level, so we can't be 100% certain of the cure yet.

**What are our treatment options?**
Right now, there isn't just one best medicine for everyone. One medication called quinine sulfate used to work very well, but it has some side effects you should be aware of (it's not for everyone). Other newer medications that help keep the nerve membranes calm seem to have the same result with fewer side effects, but they aren't all proven yet either.

**The Bottom Line:**
This is likely just a common condition, and while it doesn't sound like a life-threatening emergency, the pain makes a big difference in your quality of life. We are currently looking at more studies to find medicines that work well without causing any serious side effects. Until then, we will probably just try some different treatments with caution.

Would you like to ask me anything about what this might mean for your daily life?

## 30838456 Serotonin and Norepinephrine Reuptake Inhibitors (15.01 s)

Hello! I'm here to help you understand your treatment options for depression and related pain conditions. Let's break down this clinical note into simple terms.

### What is this medication?
This drug belongs to a specific group called **Serotonin and Norepinephrine Reuptake Inhibitors (SNRIs)**. Think of the "neurons" in your brain as post offices that send messages between areas of your body. Sometimes, people's mental health gets stuck when these messengers don't get sent properly.

This medication works by increasing the supply of these messages. Specifically, it helps with two types of messages: **serotonin** (related to mood and sleep) and **norepinephrine** (related to focus, energy, and movement).

### A Special Detail: Duloxetine
There is one special version of this drug mentioned here called **Duloxetine**. While other medications in this class work generally well for many people, Duloxetine stands out because it has a "balanced profile."
*   **Balance:** It binds to the transporters that carry messages with an even split, roughly a **10:1 ratio** between serotonin and norepinephrine. This balance helps regulate your mood and physical symptoms together, rather than just fixing one at a time.
*   **Caution with interactions:** Because this drug affects how certain other drugs are broken down in your liver (specifically the enzyme CYP2D6), if you take it with some other medicines that rely heavily on this process, your levels might change too much. The doctor will need to watch closely and may need to adjust your dose slightly.
*   **Side Effects:** The most common things people experience include nausea (feeling sick), dry mouth, dizziness, constipation, trouble sleeping at night, a feeling of lack of energy, and sometimes a temporary rise in blood pressure.
*   **Effectiveness:** Doctors have found that Duloxetine works just as well for treating **Major Depressive Disorder (MDD)** as some other popular antidepressants. It is actually FDA-approved specifically to treat pain caused by diabetes or nerve damage from injuries. It also helps with chronic fibromyalgia and musculoskeletal pain (like back or joint pain).
*   **Important Warning:** Even though the note talks about "class" drugs, there are three versions of this group: Sertraline, Paroxetine, and Fluoxetine. **All three versions have a risk for a serious, potentially life-threatening condition called Serotonin Syndrome** if combined with certain other medications (like MAOIs). This means you should never take an antidepressant from *any* class in this family while on one that affects these messages. Your doctor will make sure no such combination occurs.

### The Bottom Line
If you are looking at this medication, remember that it is designed to treat both how your brain feels (mood) and how your body moves or feels pain. While it has a slight edge in balancing its effects over time, the medical team must ensure you do not take it while on other antidepressants, as there is always a small risk of an interaction with specific drugs.

*Disclaimer: I am an AI assistant. This explanation is for educational purposes based on the text you provided and does not constitute medical advice. Always follow the instructions given directly by your prescribing physician.*

## 16445274 Hyperkalemia (12.92 s)

Hello! I've read this medical note, and I'm glad to explain it simply in plain English. Think of your **potassium** as the main electrical battery in your heart that makes it beat properly. When that level gets too high (called *hyperkalemia*), it can stop the heart from working normally—and that's serious, but manageable.

Here is what the note is telling us:

**1. What caused it?**
The note mentions a few common ways potassium levels rise in your body:
*   **Medications:** Some drugs change how your kidneys handle potassium, and others can directly stop potassium from getting inside your cells where it's needed most for heart function.
*   **Illness or Dehydration:** Being sick or having low fluid levels often leads to an inability to get rid of potassium through your urine.
*   **Diabetes (Long-term):** In people with long-term kidney issues from diabetes, a specific hormonal mismatch can cause this. Your kidneys aren't making the necessary hormone to move potassium into your cells, which builds up.

**2. Is it dangerous right now?**
Yes, if you have **typical signs or symptoms of heart trouble**—like an abnormal heartbeat or irregular rhythm—this is considered life-threatening. The note notes that a sudden jump in potassium numbers confirms this risk.

**3. What do we check next?**
Before deciding on complex treatments, doctors want to see your:
*   Urine potassium levels
*   How well your kidneys are filtering waste (creatinine)
*   Your body's water balance (osmolarity)
This information will help us figure out the root cause and tailor our treatment effectively.

**4. What treatments might be needed?**
The plan depends on why it happened, but generally includes two main strategies:

*   **Immediate Protection (While Potassium is High):**
    *   We may start with **Calcium IV**. This acts like a shield for your heart right now to stop dangerous rhythms from happening. *Note: This doesn't actually lower the potassium level in your blood.*
    *   We can also give **Insulin and Glucose**, or use inhalers called "beta-agonists." These work fast to push potassium out of your cells into your urine rapidly.

*   **Long-term Fix (After Stabilizing):**
    *   Once the heart is safe, we'll focus on lowering the total amount of potassium in your body using methods like dialysis or specific medications, depending on why it started in the first place.

**Bottom Line:** The condition itself isn't necessarily fatal if caught early and treated properly. However, because high potassium affects how well your heart beats, our priority is to protect your heartbeat while we sort out the cause.

I hope this clarifies things for you! Let me know if you have any other questions.

## 9803200 Diagnosis and classification of diabetes mellitus: new criteria (14.47 s)

Hey there! I want to explain what these new medical guidelines mean for your care, written exactly in plain English so you can easily understand and act on it.

Here's the bottom line: **The way we define and treat diabetes is getting simpler, more accurate, and earlier.**

Think of this as a fresh "rulebook" for doctors to catch people with high sugar levels sooner, before they get hurt. Here is exactly what changed and why it matters for you:

**1. The Names have Changed (IDDM vs. Type 2)**
Doctors used to use old abbreviations like IDDM (Induced Diabetes Mellitus) or NIDDM (Nondiabetic Diabetes Mellitus). We are switching to the standard terms **"Type 1"** and **"Type 2."**
*   **What does this mean for you?** It means we treat these two types of diabetes with different strategies. Since there are two types, doctors now know which one your patient has better.

**2. Diagnosis is Simpler**
In the past, it took three blood tests to diagnose diabetes. Now, it requires just **two**. If a fasting blood sugar test shows a level of **126 mg/dL (7 mmol/L) or higher**, that confirms the diagnosis. This makes the process quicker and less confusing for patients.

**3. When to Check Up: Starting at 45**
*   **For most people:** You should start screening for diabetes at age **45**. Then, every few years (every three years), unless you have specific risk factors.
*   **Why earlier?** Because if we find high sugar when someone is younger or has no symptoms, it's often easier to keep blood sugar in check and stop serious complications before they happen.

**4. Who Needs You to Watch More Closely?**
Not everyone needs to start the screening at 45. If you (or your family) have any of these risk factors, doctors should look for high blood sugar much earlier:
*   Being overweight or obese.
*   Having a first-degree relative (like a parent, sibling, or child) who has diabetes.
*   High blood pressure (hypertension).
*   High levels of triglycerides (hypertriglyceridemia).
*   A history of high insulin resistance (impaired glucose homeostasis).

**5. The Big Benefit: Preventing Trouble Later**
The most important part of these updates is the outcome. Because we are detecting diabetes earlier and using more accurate tools:
*   We can help keep blood sugar levels tight.
*   This reduces the risk of major health problems in the future, including heart issues, eye damage, and nerve damage.

**Summary Checklist for You:**
*   **Age 45 or older:** Ask your doctor about checking for diabetes.
*   **Family History:** If you know a relative has diabetes, discuss screening sooner than usual.
*   **Lifestyle:** Maintaining healthy weight helps the blood sugar levels drop naturally and reduces the risk of complications.

You've got this! Early detection gives you the best chance to stay healthy long-term.

## 21636864 Arthroscopic treatment of popliteal cyst (12.27 s)

Hello! I'm here to explain your medical chart in simple terms so you feel comfortable about your care. This document tells us what doctors did, how well it worked, and what they learned from a group of patients over a specific time period.

Here is a breakdown in plain language:

**What Was Treated?**
About 11 people visited our clinic between July 2007 and July 2009 who had a popping or swelling feeling near the back of their knee (the popliteal cyst). Before we started, each person got an MRI to make sure it was that specific condition and to check for other issues in the joint. Doctors looked for two things while treating:
1. A small valve in the knee that lets fluid go in and out (called a "valvular opening").
2. Any damage to the cartilage or meniscus inside the joint.

**What Did They Do?**
They used arthroscopy—a very tiny, painless surgical technique with a camera—to remove the cysts. In addition to taking out the cyst, they fixed any other problems they found, such as torn meniscus or broken cartilage. They also took steps to close off that valve so fluid doesn't leak back into the joint.

**How Did It Work?**
*   **Results:** Almost everyone improved immediately after the surgery. The swelling and popping stopped for them.
*   **Long-term Safety:** After a few months or years, only a very small number of people (a "low" recurrence rate) came back with new cysts.
*   **Risks:** There were no major complications like infection or bleeding. Interestingly, they noticed that one patient had the valve open and didn't know about it; unfortunately, they missed correcting this issue.

**What Do They Conclude?**
Based on these 11 patients, the doctors are confident that arthroscopic treatment is **safe and effective**. Fixing the cysts helps symptoms disappear quickly. However, because of one case where the valve wasn't corrected (even though it caused no bad events in the others), they now know it's important to check for and correct those valves properly during surgery to improve long-term results.

**Is There Anything You Need to Do?**
*   **Immediate:** Nothing needs to be done right now, as this was a past study of successful treatments.
*   **Future:** If you ever feel a pop in the back of your knee again and it sounds like fluid draining, please come back immediately. Early treatment usually means faster relief.

Would you like to know anything specific about why we recommend seeing us again if symptoms return?

## 16127951 Hyperthyroidism: diagnosis and treatment (18.4 s)

Hi! I'm here to help you understand this medical note so it sounds easier to work with. Let's break down what that text actually means for your care plan.

At its core, this note is explaining **how doctors figure out why your thyroid isn't working properly** and **what they will do about it.**

### 1. How Doctors Figure It Out
The first part of the note says they need to know two things:
*   **What's causing the problem?** Is it something specific like a virus (thyroiditis), a build-up of iodine in your thyroid, or a medication affecting you? Or is it called **Graves' disease**? That's often the most common cause.
*   **How do we find out?** We start by measuring how much "TSH" makes itself. If that number is low, we get worried something is going wrong. Sometimes, if that isn't clear, doctors use a special scan (like the radionuclide uptake test) to see which of these possibilities it is.

### 2. The Treatment Options
This is probably the most important part for you to understand: **the treatment depends entirely on the specific cause of your condition.**

*   **If it's transient hyperthyroidism (caused by thyroiditis):** The note tells us that treating your symptoms usually isn't necessary because the thyroid will naturally calm down on its own in a few weeks.
*   **If it's Graves', toxic multinodular goiter, or toxic adenoma:** These are conditions where the thyroid is "overstimulated." There are three main ways doctors treat these:
    1.  **Radioactive Iodine:** This is the **most common treatment** in the US for people who don't have side effects from other medicines. It's like using a radioactive tracer that is safely absorbed by your thyroid to stop it from working (though you lose some of your iodine, which helps the overactive thyroid).
    2.  **Medications:** Drugs specifically designed to block the thyroid from taking in too much hormone.
    3.  **Surgery:** Removing a portion or all of your thyroid gland. Doctors might recommend this if radioactive iodine doesn't work for you, if you have other conditions that make the radiation unsafe, or if the tumor is squeezing on your airway.

### 3. Special Care Groups
The note reminds us to pay special attention to certain groups of patients:
*   **Pregnant and breastfeeding women:** Thyroid hormones pass through milk or are needed in very high levels during pregnancy, so doctors must be extra careful with drug dosages or choosing between surgery/radiation vs. medication.
*   **Eye-related Graves' disease (Ophthalmopathy):** If your condition is causing eye irritation or swelling, the treatment approach might need to differ to protect your eyes.
*   **Medication Side Effects:** People taking certain heart medications (like amiodarone) are at higher risk for hyperthyroidism from side effects. This requires careful monitoring.

### 4. Your Input Matters
Finally, the note emphasizes that every patient is different. Doctors need to weigh **your personal desires** (for example, how much surgery you want versus how long you can tolerate pills). They also use **dose monitoring** to make sure your treatment plan stays safe over time.

**In short:** The best treatment for hyperthyroidism depends on *why* your thyroid is reacting this way. For many common cases, radioactive iodine is the go-to, but if you're pregnant, have eye issues, or have other health conditions, we will tailor the specific plan to keep your health safe and happy.

Does that make sense? If there are any specific symptoms you are worried about right now, please tell me, and I can help you figure out how they relate to this note!

## 15156437 Meta-analysis of cephalosporins versus penicillin for treatment of group A streptococcal tonsillopharyngitis in adults (13.35 s)

Here is an explanation of what that medical note means, broken down into simple terms:

### The Bottom Line
The study concludes that **oral antibiotics called cephalosporins are generally more effective than the older antibiotic penicillin** for treating tonsillopharyngitis (inflammation of the throat and back of the tongue caused by Group A Streptococcus bacteria). Specifically, it is about twice as likely to get better if you use a cephalosporin instead of penicillin.

### Breaking It Down
Think of this meta-analysis as an expert team that put together information from 9 separate studies. Each study involved over **2,000 patients**. The data showed:

1.  **Success Rates are Better with Cephalosporins:**
    *   **Bacteriologic Cure (Getting rid of the bacteria):** If a patient's throat infection is actually caused by those specific bacteria and they aren't fully cured after treatment, they can get it again later. The note says this risk is almost **double** if you take penicillin compared to cephalosporins.
    *   **Clinical Cure (Feeling Better):** Patients feeling better are 2.29 times more likely to use a cephalosporin than a penicillin (with a very high level of confidence).

2.  **The "Why" Matters (Important Context):**
    The study added some "sensitivity analyses" which looked at different versions of the studies to make sure the results were accurate. Even when they checked for:
    *   Whether the doctors were double-blinded (neither patient nor doctor knew which drug was being used).
    *   High-quality trials with clear instructions.
    *   Testing the throat again 3 to 14 days after starting treatment (Test-of-cure),
    
    **The result stayed the same.** The data consistently showed that cephalosporins are better than penicillin for curing these infections.

### What This Means for You
*   **Efficacy:** If your doctor has chosen penicillin specifically because they believe it works best in your specific case, you can rest easy knowing that the newer cephalosporin options work about twice as well to clear the infection completely and prevent symptoms from returning.
*   **Not a Replacement for All Decisions:** While this data strongly favors cephalosporins, every patient is unique. Your doctor will consider factors like your specific bacterial resistance patterns or allergies before prescribing either type of antibiotic.

**Note on Penicillin:** It is worth mentioning that many people are now allergic to penicillin, which is why doctors often prefer cephalosporins as the alternative first-line therapy in many guidelines.

## 1000139 Clozapine versus perphenazine: the value of the biochemical mode of action of neuroleptics in predicting their therapeutic activity (9.91 s)

Hello! I'm here to help you understand what this clinical note is really telling us about how certain medicines work for mental health issues. Let's break it down simply.

**The Big Question:** Does the exact shape or chemical makeup of a medicine tell us exactly how it will treat someone?
*(Note: The word "relably" in the text was likely meant to be "reliably.")*

**The Study They Did:**
Doctors ran a special study where two medicines were compared under blind conditions (meaning neither the doctors nor the patients knew which drug was being tested). They looked at how well **Clozapine** and **Perphenazine** treated people with serious mental illnesses (psychoses) in different stages.

**The Key Finding:**
Here is the simple truth they discovered:
*   **Clozapine** acts mostly like it blocks brain signals that use "Nerve Growth Factor" (NA) messengers. It does *not* block those same signals for dopamine (DA).
*   **Perphenazine** acts mostly like it blocks dopamine (DA) messengers. It does *not* block Nerve Growth Factor as strongly.

**So, what does this mean for you?**
Because of these different chemical behaviors, the study showed that the medicine works in a very specific way:
*   Since Clozapine affects "Nerve Growth Factor" messages so much, it can actually be quite strong in stopping symptoms and sedative (calming) effects.
*   Since Perphenazine mainly hits dopamine messages, its therapeutic effect is linked to how well that specific pathway functions.

**The Conclusion:**
Basically, while you could guess the medicine's chemical makeup, doctors proved that if we know how the drug actually affects the brain's electrical and chemical signals in real life (its biochemical action), we have a much stronger predictor of whether it will work well for you.

**In one sentence:**
It turns out that **what the medicine does inside your body matters more than its chemical shape**, because that determines exactly how it calms your mind and treats your symptoms.

## 7698623 Management of foreign bodies of the upper gastrointestinal tract: update (12.92 s)

Hello! I understand the situation, and please don't worry—we are taking care of this very serious matter.

Here is a plain-language summary of what happened based on that note:

**What went wrong?**
We found **242 small foreign objects** (like buttons, coins, metal shards, or bits of meat) stuck up your esophagus (the tube holding food in your chest) and some further down into your stomach. These are all very dangerous because they can hurt your heart if not removed quickly.

**How did we treat it?**
We used a special flexible camera called a "panendoscope" to look inside and carefully remove the objects. This method is extremely safe for you, so only **0.4%** of your cases needed surgery later on. Fortunately, there were **no side effects or problems** (like nausea or blood) during the procedure.

**How long do things last?**
Here is what usually happens:
*   **In your throat:** If a coin or button was in your upper part of the esophagus, it gets removed immediately to prevent blockage. If it was lower down, we tried to take it out within 12 hours.
*   **In your stomach:** If an object got stuck in your stomach, most will just pass through on their own without needing more work. However, if a meatball or piece of bone is there blocking food from going up, that is an urgent problem that must be removed quickly to avoid choking or starvation.
*   **Batteries are emergency:** Any button battery found in the esophagus needs to be removed right away because they damage tissue and can kill cells if left too long.

**Is it safe?**
Yes! This treatment was done without anesthesia in 75% of your cases (25%), which is the safest possible way for you while keeping us from being overly aggressive with sedation. For the remaining patients, we used breathing tubes to protect your airway while the doctors worked.

**What else can you expect?**
Because so many items were small and sharp (like coins or metal shards), our team had to be very careful. We sometimes let the doctors try moving them again in a simulation to ensure we would get them out safely without hurting your throat or tearing any tissue during the real removal. This step helps make sure you don't have a problem later.

**Why is this important?**
Modern technology has made removing these objects much easier and safer, so most of them are taken out in one visit with good results. We just need to watch your throat closely for the next day or two to make sure nothing gets caught.

I hope you understand now and feel better. Is there anything specific you would like to ask me about this?

## 3068693 Finger pain (18.2 s)

Here is an explanation of that clinical note broken down into plain language, keeping things simple so you don't have to memorize complex medical terms.

### A Simple Summary for You
This note explains how doctors figure out why your fingers hurt. While many finger pain cases just need rest and time, some serious causes (like blood flow issues or infections) can become very dangerous if they aren't treated quickly. The note also suggests that if you're still feeling bad after a few weeks, seeing an Occupational Therapist might help keep your hand moving and working better.

---

### Detailed Breakdown

Here is what that text is really telling you, in everyday terms:

**1. Most finger pain comes from common, fixable reasons.**
*   **Most cases:** Your fingers are hurting because of a tiny injury (like dropping something on them). These are usually fixed quickly with rest and exercise.
*   **What helps:** Ice, keeping the hand still to stop bleeding/swelling, and gentle movement.

**2. Watch out for serious infections.**
*   If your finger pain is caused by an infection (bacteria or viruses), it can spread through the whole body if not treated right away.
*   Treatments include taking antibiotics (medicine) or draining any pockets of pus on the skin.
*   **When to worry:** If the doctor suspects a serious infection, they might call in a specialist (like a hand surgeon or rheumatologist) to help manage it.

**3. Some pain means "something is wrong" right now.**
*   **Vascular and Ischemic causes:** These involve blood flow issues that affect tissue viability (life). If you are experiencing sudden, severe pain that feels different from usual finger pain, this could be a life-or-death emergency requiring immediate medical attention.

**4. Common nerve-related pain.**
*   Many times, your fingers hurt because the nerves to your hand are pinched or irritated.
*   The note mentions **Carpal Tunnel Syndrome**, which is one of the most common causes of this type of pain. If symptoms don't improve with rest, seeing a specialist might help relieve pressure on your nerve.

**5. Is it something else entirely?**
*   Sometimes finger pain isn't actually about your fingers at all. It could be related to systemic conditions (like Rheumatoid Arthritis or Vasculitis) or even early signs of major illnesses like **Hypertrophic Pulmonary Osteoarthropathy**. These often look for help from a **rheumatologist** to find the root cause.
*   Sometimes, your brain signals pain in an area of your body where there isn't any actual inflammation. This is called "reflex sympathetic dystrophy" (sometimes known as phantom limb pain), caused by how our nerves connect, and usually treated with a specialist's help.

**6. Don't let symptoms disappear!**
*   Even if you just need a finger taped up for a couple of days, the note advises: **"Certain infectious, traumatic, and ischemic causes must be diagnosed promptly to avoid significant morbidity."**
    *   *Meaning:* While it might sound silly to tell your doctor "wait, let me wait another two hours," these types of injuries can get complicated or permanent if they aren't caught quickly. The note suggests that getting an expert opinion (referral) speeds up the recovery process and prevents long-term damage.

### How You Can Use This
*   **If you hurt your fingers:** Ask your doctor, "Could this be a simple injury, an infection, or a nerve problem?"
*   **If pain lasts more than 2–3 days without getting better:** Go back to the doctor because some causes require specific treatment sooner than others.
*   **After a minor fix:** Mention that seeing an **Occupational Therapist** (who helps people get their hands/worked in again) might help you feel like your hand is working better once it's healed.

***Disclaimer:** I am an AI assistant, not a doctor. This explanation is for educational purposes and should not replace the specific advice given by the medical professionals who reviewed this note.*

In [12]:
# Mistakes in each output, from reading it against the abstract and the human version (same as Notes below).
notes = {
    "15902691": [
        "Says cramps are likely caused by your medical history, exercise, or blood tests, but the abstract says history, exam, and labs help find the cause.",
        "Talks about the reader's own case and plan (\"your specific case\", \"we will try treatments\") even though it's a review article.",
        "Adds that doctors will \"listen to what sounds when you move\", which isn't in the abstract.",
    ],
    "30838456": [
        "Names sertraline, paroxetine, and fluoxetine as the drugs in this class, but those are SSRIs, not SNRIs.",
        "Calls the 10:1 binding ratio \"an even split\".",
        "Says to \"never take an antidepressant from any class\" with another, but the abstract only warns about combining with MAOIs.",
        "Adds neuropathic pain \"from injuries\" and a \"temporary\" rise in blood pressure.",
    ],
    "16445274": [
        "Says insulin + glucose and beta-agonists push potassium \"out of your cells into your urine\", but they move potassium into cells.",
        "Explains the diabetes cause as a hormone that moves potassium into cells, but it actually lowers how much potassium the kidneys get rid of.",
        "Says \"signs or symptoms of heart trouble\" make it life-threatening, but the abstract says typical ECG changes or a fast rise in potassium.",
        "Adds dialysis and leaves out sodium polystyrene (with furosemide and saline) for lowering total body potassium.",
    ],
    "9803200": [
        "Spells out IDDM as \"Induced Diabetes Mellitus\" and NIDDM as \"Nondiabetic Diabetes Mellitus\", but they mean insulin-dependent and non-insulin-dependent.",
        "Says \"in the past it took three blood tests\", which the abstract doesn't say.",
        "Leaves out that only half of people with diabetes have been diagnosed.",
        "Adds that keeping a healthy weight makes blood sugar \"drop naturally\".",
    ],
    "21636864": [
        "Says the valve is closed so fluid \"doesn't leak back into the joint\", but the valve lets fluid flow from the joint into the cyst.",
        "Says \"one patient had the valve open and didn't know about it\", but the doctors failed to find the valve opening in one patient.",
        "Talks to the reader as a patient (\"visited our clinic\", \"come back immediately\") even though it's a study of 11 patients.",
        "Adds a \"popping\" feeling, \"painless\" surgery, and infection or bleeding as examples of complications.",
    ],
    "16127951": [
        "Says for thyroiditis \"treating your symptoms usually isn't necessary\", but the abstract says treating the symptoms is usually enough.",
        "Calls thyroiditis \"a virus\" and adds \"a build-up of iodine\" as a cause.",
        "Says radioactive iodine is for people \"who don't have side effects from other medicines\", but it means people with no reason they can't have it.",
        "Says antithyroid drugs \"block the thyroid from taking in too much hormone\", but they stop it from making hormone.",
        "Leaves out that new therapies are under investigation.",
    ],
    "15156437": [
        "Says \"each study involved over 2,000 patients\", but 2,113 is the total across all 9 trials.",
        "Says \"patients feeling better are 2.29 times more likely to use a cephalosporin\", which is backwards: cephalosporins give 2.29x the odds of cure.",
        "Says \"if your doctor has chosen penicillin... you can rest easy knowing cephalosporins work twice as well\", which contradicts itself.",
        "Adds penicillin allergy and resistance patterns.",
    ],
    "1000139": [
        "Explains NA as \"Nerve Growth Factor\", but NA is noradrenaline (norepinephrine).",
        "Says \"doctors proved\" it, but the abstract says the hypothesis was shown to be plausible.",
        "Leaves out the prediction that perphenazine is the stronger antipsychotic and weaker sedative.",
        "Comments on a typo in the text, which isn't useful to a patient.",
    ],
    "7698623": [
        "Says 242 objects were found \"stuck up your esophagus\", but it's a report on 242 cases, not one patient.",
        "Puts the meat blockage in the stomach, but meat impaction is in the esophagus.",
        "Says \"without anesthesia in 75% of your cases (25%)\", which is garbled.",
        "Adds that objects \"can hurt your heart\" and to watch your throat \"for the next day or two\".",
        "Leaves out that 98.8% were removed with an endoscope and that button batteries usually pass once in the stomach.",
    ],
    "3068693": [
        "Calls reflex sympathetic dystrophy \"phantom limb pain\", but they're different conditions.",
        "Says minor injuries need ice and keeping the hand still \"to stop bleeding\", but the abstract says immobilize, then exercise.",
        "Adds going back if pain lasts \"2-3 days\", seeing an OT after \"a few weeks\", and a rheumatologist for infections (the abstract says hand surgeon).",
        "Leaves out the specific infections (cellulitis, paronychia, felon) and referral to a neurologist.",
    ],
}

### Notes

- 10–18 s per abstract. Outputs are 1.6–5x longer than the human versions (314–637 words vs. 72–362).

Mistakes I found by reading each output against its abstract and the human version

15902691 Muscle cramps
- Says cramps are likely caused by your medical history, exercise, or blood tests, but the abstract says history, exam, and labs help find the cause.
- Talks about the reader's own case and plan ("your specific case", "we will try treatments") even though it's a review article.
- Adds that doctors will "listen to what sounds when you move", which isn't in the abstract.

30838456 SNRIs (duloxetine)
- Names sertraline, paroxetine, and fluoxetine as the drugs in this class, but those are SSRIs, not SNRIs.
- Calls the 10:1 binding ratio "an even split".
- Says to "never take an antidepressant from any class" with another, but the abstract only warns about combining with MAOIs.
- Adds neuropathic pain "from injuries" and a "temporary" rise in blood pressure.

16445274 Hyperkalemia
- Says insulin + glucose and beta-agonists push potassium "out of your cells into your urine", but they move potassium into cells.
- Explains the diabetes cause as a hormone that moves potassium into cells, but it actually lowers how much potassium the kidneys get rid of.
- Says "signs or symptoms of heart trouble" make it life-threatening, but the abstract says typical ECG changes or a fast rise in potassium.
- Adds dialysis and leaves out sodium polystyrene (with furosemide and saline) for lowering total body potassium.

9803200 Diabetes diagnosis
- Spells out IDDM as "Induced Diabetes Mellitus" and NIDDM as "Nondiabetic Diabetes Mellitus", but they mean insulin-dependent and non-insulin-dependent.
- Says "in the past it took three blood tests", which the abstract doesn't say.
- Leaves out that only half of people with diabetes have been diagnosed.
- Adds that keeping a healthy weight makes blood sugar "drop naturally".

21636864 Popliteal (Baker's) cyst
- Says the valve is closed so fluid "doesn't leak back into the joint", but the valve lets fluid flow from the joint into the cyst.
- Says "one patient had the valve open and didn't know about it", but the doctors failed to find the valve opening in one patient.
- Talks to the reader as a patient ("visited our clinic", "come back immediately") even though it's a study of 11 patients.
- Adds a "popping" feeling, "painless" surgery, and infection or bleeding as examples of complications.

16127951 Hyperthyroidism
- Says for thyroiditis "treating your symptoms usually isn't necessary", but the abstract says treating the symptoms is usually enough.
- Calls thyroiditis "a virus" and adds "a build-up of iodine" as a cause.
- Says radioactive iodine is for people "who don't have side effects from other medicines", but it means people with no reason they can't have it.
- Says antithyroid drugs "block the thyroid from taking in too much hormone", but they stop it from making hormone.
- Leaves out that new therapies are under investigation.

15156437 Strep throat antibiotics
- Says "each study involved over 2,000 patients", but 2,113 is the total across all 9 trials.
- Says "patients feeling better are 2.29 times more likely to use a cephalosporin", which is backwards: cephalosporins give 2.29x the odds of cure.
- Says "if your doctor has chosen penicillin... you can rest easy knowing cephalosporins work twice as well", which contradicts itself.
- Adds penicillin allergy and resistance patterns.

1000139 Clozapine vs. perphenazine
- Explains NA as "Nerve Growth Factor", but NA is noradrenaline (norepinephrine).
- Says "doctors proved" it, but the abstract says the hypothesis was shown to be plausible.
- Leaves out the prediction that perphenazine is the stronger antipsychotic and weaker sedative.
- Comments on a typo in the text, which isn't useful to a patient.

7698623 Swallowed foreign bodies
- Says 242 objects were found "stuck up your esophagus", but it's a report on 242 cases, not one patient.
- Puts the meat blockage in the stomach, but meat impaction is in the esophagus.
- Says "without anesthesia in 75% of your cases (25%)", which is garbled.
- Adds that objects "can hurt your heart" and to watch your throat "for the next day or two".
- Leaves out that 98.8% were removed with an endoscope and that button batteries usually pass once in the stomach.

3068693 Finger pain
- Calls reflex sympathetic dystrophy "phantom limb pain", but they're different conditions.
- Says minor injuries need ice and keeping the hand still "to stop bleeding", but the abstract says immobilize, then exercise.
- Adds going back if pain lasts "2-3 days", seeing an OT after "a few weeks", and a rheumatologist for infections (the abstract says hand surgeon).
- Leaves out the specific infections (cellulitis, paronychia, felon) and referral to a neurologist.

Patterns

The prompt doesn't fit PLABA. It says "clinical note to the patient", so the model treats research abstracts as the reader's own chart ("242 objects stuck up your esophagus", "visited our clinic"). Wrong explanations of jargon are still the biggest problem: IDDM/NIDDM spelled out wrong, NA explained as "Nerve Growth Factor", reflex sympathetic dystrophy called phantom limb pain, and SSRIs listed as SNRIs. Some things are flipped backwards: insulin pushes potassium out of cells (it's into), symptom treatment "isn't necessary" (it's usually enough), and "more likely to use a cephalosporin" (more likely to be cured). Adding things not in the abstract happened in all 10, with made-up timings, causes, and advice to the reader. Numbers are mostly right (126 mg/dL, 2.29, 0.4%, 12 hours, 11 patients), so the mistakes are about meaning more than numbers.

## Second model: gemma2:2b (speed)

In [13]:
# Same 10 PLABA abstracts and prompt on gemma2:2b, to compare speed with qwen3.5:2b.
SECOND_MODEL = "gemma2:2b"

# Warm-up call, so loading the model into memory isn't counted in the first abstract's time.
call_model(system, "Hi", model=SECOND_MODEL)

gemma_results = []

for ex in examples:
    start = time.time()
    output = call_model(system, prompt + ex["text"], model=SECOND_MODEL)
    seconds = round(time.time() - start, 2)

    gemma_results.append({"id": ex["id"], "title": ex["title"], "seconds": seconds, "output": output})
    print(ex["id"], "took", seconds, "s")

15902691 took 5.52 s


30838456 took 5.87 s


16445274 took 6.46 s


9803200 took 6.8 s


21636864 took 5.19 s


16127951 took 8.05 s


15156437 took 5.23 s


1000139 took 6.41 s


7698623 took 5.86 s


3068693 took 5.89 s


In [14]:
# Speed: seconds per abstract, and words per second (longer answers take longer).
qwen = pd.DataFrame(plaba_results)
gemma = pd.DataFrame(gemma_results)

compare = pd.DataFrame({
    "id": qwen["id"],
    "qwen_seconds": qwen["seconds"],
    "gemma_seconds": gemma["seconds"],
    "qwen_words": qwen["output"].str.split().str.len(),
    "gemma_words": gemma["output"].str.split().str.len(),
})
compare["qwen_words_per_s"] = (compare["qwen_words"] / compare["qwen_seconds"]).round(1)
compare["gemma_words_per_s"] = (compare["gemma_words"] / compare["gemma_seconds"]).round(1)
compare.loc["average"] = compare.drop(columns="id").mean().round(1)
compare

,id,qwen_seconds,gemma_seconds,qwen_words,gemma_words,qwen_words_per_s,gemma_words_per_s
0,15902691,11.77,5.52,347.0,240.0,29.5,43.5
1,30838456,15.01,5.87,509.0,234.0,33.9,39.9
2,16445274,12.92,6.46,432.0,284.0,33.4,44.0
3,9803200,14.47,6.80,463.0,269.0,32.0,39.6
4,21636864,12.27,5.19,414.0,209.0,33.7,40.3
5,16127951,18.40,8.05,587.0,303.0,31.9,37.6
6,15156437,13.35,5.23,385.0,213.0,28.8,40.7
7,1000139,9.91,6.41,314.0,265.0,31.7,41.3
8,7698623,12.92,5.86,450.0,248.0,34.8,42.3
9,3068693,18.20,5.89,637.0,246.0,35.0,41.8


In [15]:
for r in gemma_results:
    display(Markdown(f"## {r['id']} {r['title']} ({r['seconds']} s)\n\n{r['output']}"))

## 15902691 Muscle cramps (5.52 s)

Okay, so your doctor wrote you a note about muscle cramps. Here's what it means:

* **Muscle cramps are painful, sudden spasms:** Imagine you're stretching and suddenly you feel a sharp, painful contraction in your muscle. That's a muscle cramp. 
* **Sometimes, they feel different from other muscle pain:** This is important because sometimes other aches and pains aren't exactly the same as muscle cramps. 
* **Finding out why is tricky:** To figure out what's causing your cramps, your doctor might look at things like your medical history (past illnesses, medications, etc.) and how you feel physically (examining your muscles, etc.) and run a quick blood test.
* **Not all cramps are serious:** Even though muscle cramps can feel uncomfortable, they're usually not a big deal. Doctors know that.
* **Treatment options are still being figured out:**  Doctors have some ideas about how to help, but sometimes these are just a bit of trial and error. One thing that might help is quinine sulfate, but it can also have some side effects.
* **Research is still needed:** There's lots more we don't know about why muscle cramps happen and how to treat them better. 

**Basically, your doctor is trying to figure out why you're experiencing muscle cramps and how to help you feel better.** They're also learning more about muscle cramps so they can help even more people in the future. 


Is there anything else you'd like me to explain? 


## 30838456 Serotonin and Norepinephrine Reuptake Inhibitors (5.87 s)

Okay, so let's talk about a type of medication called antidepressants.

**What is it?**

It's a class of medications that, simply put, help balance the brain chemicals called serotonin and norepinephrine.  

**How it works:**

* **Duloxetine is a "reuptake inhibitor"**: Think of it like a key fitting into a lock. It helps get more serotonin and norepinephrine into the brain.  
* **It's a "balanced" inhibitor**: This means it affects these chemicals in a pretty even way (roughly 10:1 ratio). 
* **It can affect other chemicals in the body**: It also slows down the breakdown of some chemicals in your body (CYP2D6), so it's important to be careful if you're also taking certain medications that might interact.

**Common side effects:**

* Nausea
* Dry mouth 
* Dizziness
* Constipation
* Insomnia
* Feeling tired (asthenia) 
* High blood pressure

**Why is it helpful?**

* It's been tested in many patients with depression, and it shows comparable results to other popular antidepressants.
* It's also been approved for other conditions like diabetic neuropathy, fibromyalgia, and musculoskeletal pain. 

**Important note:**

* These medications can sometimes cause a serious reaction called "serotonin syndrome" if you take them with certain other medications. It's best to talk to your doctor before starting or changing medications. 



Let me know if you have any questions about this! We can discuss your specific situation and any other concerns you have about your treatment. 


## 16445274 Hyperkalemia (6.46 s)

Okay, so let's talk about your potassium levels.  

**What's going on?**

Your body has a balance system for keeping the right amount of potassium in your blood.  Sometimes, things can go wrong and you might have too much potassium in your blood (that's called hyperkalemia).  This can be dangerous because it can affect your heart rhythm.

**Why does it happen?**

It can be caused by a few things:

* **Your kidneys not working well:** Your kidneys usually filter out excess potassium from your blood. If they're not working properly, they might not be able to get rid of it all.
* **Your body not moving potassium from your blood to your cells:** Your body has special ways to get potassium into your cells, and sometimes these mechanisms can get messed up.
* **Combination:**  It can be a mix of these problems.

**When is it a problem?**

Sometimes when your potassium levels are high, you might see changes on your EKG, or your heart rhythm gets messed up.  This can be a sign that it's becoming dangerous.  

**What can we do about it?**

We can figure out the cause of your high potassium. We'll test your urine, blood, and maybe other things to see what's going on.

* **Fast acting solutions:** We can use medications like insulin and glucose to quickly lower your potassium levels.  
* **Long term management:**  Sometimes we need to use something like sodium polystyrene to lower your potassium levels over time.
* **Calcium for your heart:**  If you are having changes to your heart rhythm, we might give you calcium to help your heart beat normally.

We'll work together to get your potassium levels back in balance and keep you healthy!  


## 9803200 Diagnosis and classification of diabetes mellitus: new criteria (6.8 s)

Okay, so let's talk about diabetes. 

There's been a change in how we categorize and diagnose diabetes. It's now easier to understand! 

**Here's the simplified version:**

* **Type 1 Diabetes:** This is a serious type of diabetes where your body can't make insulin, a hormone that helps your body use sugar. 
* **Type 2 Diabetes:** This is a more common type of diabetes where your body doesn't use insulin very well. 

Instead of using "IDDM" and "NIDDM", we're using "Type 1" and "Type 2". That sounds much simpler, right?

**Diagnosis is also easier now:**

* We can diagnose diabetes just by looking at your blood sugar levels.  Instead of a lot of tests, we just check your blood sugar level at fasting. 
* We're also using a lower blood sugar level to confirm the diagnosis (126 mg/dL or higher). 

**Why does this matter?**

The changes help us:

* **Catch diabetes sooner:**  Early diagnosis helps us control your blood sugar and prevent complications.
* **Start treatment sooner:** We can start treatment earlier to manage diabetes better and reduce risks.


**Here's the schedule:**

* **Start screening at 45:** If you're at risk, we recommend you start screening for diabetes.
* **Repeat screenings:** We will repeat the screening every 3 years for those without risk factors.
* **More frequent screenings for risk factors:**  If you have any risk factors, such as obesity, family history, high blood pressure, or high cholesterol, we will need to test you more often.


Remember, early detection is key! With your help, we can manage your diabetes and keep you healthy. 


Do you have any questions or concerns? 


## 21636864 Arthroscopic treatment of popliteal cyst (5.19 s)

Okay, let's break down this note about your popliteal cyst surgery. 

**Basically, your doctor reviewed your case and found that arthroscopic surgery to treat your popliteal cyst was a success.**

**Here's what they did:**

* **They used a special camera to look inside your knee.** This is called arthroscopy.
* **They confirmed the diagnosis:**  They used a special MRI to see your popliteal cyst and determine if your knee cartilage was damaged and if your meniscus (a C-shaped piece of cartilage in your knee) was injured.
* **They followed the Rauschning and Lindgren criteria:** These are guidelines for evaluating the severity of your popliteal cyst and associated knee issues. 

**Results:**

* **Many patients saw improvement:**  Most of the patients who had this surgery felt better after the procedure.
* **Recurrent cases are rare:** They're not likely to reoccur.
* **They didn't fix all of the problems:** In one patient, they couldn't fix a valve in your knee that wasn't completely open, which may have affected the outcome. 

**The Takeaway:**

* **This surgery is effective and safe for treating popliteal cysts.**  Your doctor feels this kind of surgery can help you feel better. 


Let me know if you have any questions.  We can discuss anything that's unclear in the note. 


## 16127951 Hyperthyroidism: diagnosis and treatment (8.05 s)

Okay, let's break down this note about your hyperthyroidism. 

**What's Hyperthyroidism?**

Essentially, your body is producing too much of the thyroid hormone. It's like the car's fuel is running on overdrive, causing symptoms like a racing heart, anxiety, or weight loss. 

**What Causes Hyperthyroidism?**

There are a few reasons for this. 

* **Graves' disease:** This is the most common cause, and it's linked to your body's immune system going haywire. 
* **Thyroiditis:** This means inflammation of your thyroid gland. It usually causes temporary hyperthyroidism.
* **Goiter (enlarged thyroid):** It can sometimes cause hyperthyroidism.
* **Medications:** Some drugs can cause hyperthyroidism as a side effect.

**How to Figure it Out?**

* **Thyroid-Stimulating Hormone Test:** This measures the amount of thyroid hormone your body is making.
* **Radionuclide Uptake:** If the test results are unclear, this test helps us figure out what's causing the hyperthyroidism.

**Treating Hyperthyroidism**

There are a few things we can do depending on the cause:

* **Graves' disease:** This can be treated with medications (antithyroid drugs) or surgery. 
* **Thyroiditis:** We can treat this with medications to relieve symptoms.
* **Goiter:** We can try medications and sometimes surgery to shrink the thyroid gland.
* **Medications:** We need to consider the specific medication and its potential effects. 

**Important Considerations**

* **Pregnancy and Breastfeeding:** Special care is needed for these groups. 
* **Graves' ophthalmopathy:**  This is a condition that causes swelling in your eyes. 
* **Amiodarone-induced hyperthyroidism:** This occurs when a heart medication called amiodarone is taken.
* **Patient Preferences:** We always consider your wishes when choosing the best treatment plan.


 **Why Monitoring is Important**

We need to closely watch the dose of medications and make sure the treatment works well for you. This helps ensure the best possible outcome for your health.



Does this explanation make sense? Do you have any questions? 


## 15156437 Meta-analysis of cephalosporins versus penicillin for treatment of group A streptococcal tonsillopharyngitis in adults (5.23 s)

Okay, so let's talk about your treatment for your tonsillitis.

**Here's the gist:**

We compared two different antibiotics, penicillin and cephalosporin, to see which one was more effective in treating your tonsillitis (the inflammation in your throat). We looked at many studies with a lot of patients. 

**The results:**

* **Bacteriologic cure rate:** This means we looked at whether the antibiotics actually killed the bacteria causing your tonsillitis. 
    * We found that cephalosporins were **better** at getting rid of the bacteria compared to penicillin. The study showed that the odds of a successful cure with cephalosporins were almost **twice as likely** as with penicillin.
* **Clinical cure rate:** This means we looked at whether your symptoms improved after taking the antibiotics. 
    * Again, the results were in favor of cephalosporins.  It's like penicillin didn't quite get the job done as well as the cephalosporins. 
 
 **Why this matters:**

These results are very important because they show that cephalosporins are a good choice for treating your tonsillitis. The study shows that penicillin is less effective and often leads to more issues than cephalosporins. We can use this information to help you feel better faster and more effectively!



 **Do you have any questions about this information?** I want to make sure you understand your treatment options. 


## 1000139 Clozapine versus perphenazine: the value of the biochemical mode of action of neuroleptics in predicting their therapeutic activity (6.41 s)

Okay, let's break this down so you can understand it better. 

**What this clinical note is about:**

* **Drugs and how they work:**  We're talking about medications called neuroleptics, sometimes called antipsychotics. They're used to treat mental illnesses like schizophrenia.
* **A problem with prediction:**  A lot of the time, we know the structure of a drug (its chemical makeup) but that doesn't always mean we know exactly how well it'll work in a patient.
* **Focusing on the "why" of the drug:**  The note suggests we should look at the *effect* the drug has on the brain, specifically the balance of two chemical messengers, dopamine (DA) and norepinephrine (NA).
* **Two drugs, two differences:**  The note talks about clozapine and perphenazine, which are two specific neuroleptics. They were tested in patients with various mental illnesses.  The results showed that clozapine affects the balance of DA and NA differently than perphenazine.  

**What this means for you:**

* **Different approaches:**  Clozapine was found to be better at affecting certain parts of the brain, while perphenazine was found to be more effective in others.
* **Predicting the effect:**  Based on these findings, the team believes that the way a drug interacts with the brain's system (its biochemical effect) is a better predictor of how it will work in a patient than just the drug's chemical structure. 

**In simpler terms:** This research helps us understand that the *how* the drug works in the brain is a more accurate indicator of its effectiveness than just knowing what it's made of.

**Do you have any questions about any of this?** 


## 7698623 Management of foreign bodies of the upper gastrointestinal tract: update (5.86 s)

Okay, let's break down this note about your stomach and throat.

**What it means:**  You have a lot of small objects stuck in your stomach and throat.  Doctors call these "foreign bodies." 

**How they were dealt with:**

* **Endoscopy:** The doctors used a thin tube (endoscope) to look at your stomach and throat. This was the most common way to remove the objects.  
* **Surgery:** Only a tiny percentage of cases needed surgery, which was very successful with no problems (like complications). 
* **Time is Key:**  If something is stuck in your esophagus (the tube that carries food from your throat to your stomach), they act fast. Coins in the esophagus need to be removed immediately if it's in the top part or within 12 hours if it's at the bottom. Meat that's stuck in your esophagus needs to be removed ASAP as it can block the passage and cause a serious problem.

**Important to Know:** 

* **Sharp Objects:** If a sharp object is stuck, it's tricky and may need special preparation before removing it.
* **Button Batteries:** These are very dangerous if they get stuck. We need to remove them right away.
* **Panendoscope:** This is the best tool for removing these objects, as it gives doctors a better look inside the stomach and throat.

**Overall:** You've been through a lot, but you're doing great.  The good news is you can get these foreign bodies removed without complications.  

**Questions:** Do you have any questions about this information? 




## 3068693 Finger pain (5.89 s)

Okay, so let's talk about your finger pain. 

**What's going on?**

Your doctor is trying to figure out why your finger is hurting. They'll look at things like:

* **The reason for your pain:** This is important. Was it an accident? Is it related to an infection? Did you just have a weird feeling in your finger?
* **What your finger is like:** They'll check for swelling, redness, warmth, and any other signs of problems.
* **If we need extra tools:** Sometimes, we need to take X-rays or test your blood to get a better idea.

**What can we do about it?**

* **Minor injuries:** If it's something like a sprain or strain, we can usually just treat it with rest and exercises to help your finger heal.
* **Infections:** This means bacteria got into your finger, and we'll use antibiotics to fight it. Sometimes, we might also drain the affected area.
* **Urgent cases:**  There are some situations where your finger pain is a sign of something serious, like a blockage in your blood vessels. We need to address this quickly to keep your finger from losing function.

**Other things to note:**

* **Referrals:** Sometimes, your doctor might recommend seeing a specialist like a hand surgeon, rheumatologist, or neurologist if the problem is complex.
* **Therapy:**  An occupational therapist can help you regain function and adapt your daily activities. 


**Remember:** It's important to keep your doctor updated if your finger pain continues or gets worse. 


In [16]:
# Mistakes in each gemma2 output, from reading it against the abstract and the human version (same as Notes below).
gemma_notes = {
    "15902691": [
        "Treats the review article as the reader's own note (\"your doctor wrote you a note about muscle cramps\").",
        "Leaves out that cramps come from nerves and that other drugs are probably just as effective as quinine.",
    ],
    "30838456": [
        "Says duloxetine affects serotonin and norepinephrine \"in a pretty even way\" while giving a 10:1 ratio.",
        "Says serotonin syndrome happens with \"certain other medications\" instead of naming MAOIs.",
    ],
    "16445274": [
        "Leaves out that IV calcium doesn't lower potassium, and the beta-agonist option.",
        "Leaves out the common triggers (new medications, illness, dehydration) and the diabetes-related cause.",
    ],
    "9803200": [
        "Says to start screening at 45 \"if you're at risk\", but 45 is for people without risk factors; people with risk factors start earlier.",
        "Lists \"high cholesterol\" as a risk factor where the abstract says high triglycerides, and leaves out impaired glucose.",
        "Makes diagnosis sound like one fasting blood sugar check, leaving out that it takes two abnormal tests.",
    ],
    "21636864": [
        "Treats the study as the reader's own surgery (\"your doctor reviewed your case\") and never says it was 11 patients.",
        "Says \"most of the patients\" improved, but all of them did.",
        "Says they \"couldn't fix a valve in your knee that wasn't completely open\", but they failed to find the valve opening in one patient.",
    ],
    "16127951": [
        "Says the TSH test measures the thyroid hormone your body makes, but TSH is a different hormone that tells the thyroid what to do.",
        "Leaves out radioactive iodine, which the abstract says is the treatment of choice in the US.",
        "Leaves out toxic adenoma as a cause, and when surgery is used (other treatments fail or a goiter presses on the neck).",
    ],
    "15156437": [
        "Talks as if the study were the reader's own treatment (\"we compared two antibiotics... treating your tonsillitis\").",
        "Leaves out the numbers (9 trials, 2,113 patients, 2.29 odds of clinical cure) and that it's about strep throat in adults.",
        "Adds that penicillin \"often leads to more issues\".",
    ],
    "1000139": [
        "Says clozapine was \"better at affecting certain parts of the brain\" and perphenazine \"more effective in others\", instead of which signals each one blocks.",
        "Leaves out the prediction that perphenazine is the stronger antipsychotic and weaker sedative.",
    ],
    "7698623": [
        "Says \"you have a lot of small objects stuck in your stomach and throat\", but it's a report on 242 different cases.",
        "Leaves out the success rate (98.8% removed with an endoscope) and that coins and batteries usually pass once they reach the stomach.",
    ],
    "3068693": [
        "Leaves out all the nerve and whole-body causes: carpal tunnel syndrome, rheumatoid arthritis, vasculitis, HPOA, and reflex sympathetic dystrophy.",
        "Leaves out the specific infections (cellulitis, paronychia, felon).",
    ],
}

In [17]:
# Log every PLABA run (both models) to a CSV: one row per example. Output text only, no PLABA text.
rows = []

for model, results_list, model_notes in [(MODEL, plaba_results, notes), (SECOND_MODEL, gemma_results, gemma_notes)]:
    for r in results_list:
        rows.append({
            "dataset": "PLABA",
            "id": r["id"],
            "title": r["title"],
            "model": model,
            "seconds": r["seconds"],
            "notes": " | ".join(model_notes.get(r["id"], [])),
            "output": r["output"],
        })

log = pd.DataFrame(rows)
log.to_csv("../results/baseline_log.csv", index=False)
print("Saved", len(log), "rows to results/baseline_log.csv")

Saved 20 rows to results/baseline_log.csv


### Notes

- gemma2:2b took 5–8 s per abstract (average 6.1 s) vs. 10–18 s for qwen3.5:2b (average 13.9 s), about 2.3x faster.
- Part of that is shorter answers (about 250 words vs. 450), but gemma is faster per word too (41 vs. 33 words per second).
- gemma had a warm-up call and qwen didn't, but qwen's first abstract wasn't slower than the rest, so it doesn't change the comparison.

Mistakes I found by reading each output against its abstract and the human version

15902691 Muscle cramps
- Treats the review article as the reader's own note ("your doctor wrote you a note about muscle cramps").
- Leaves out that cramps come from nerves and that other drugs are probably just as effective as quinine.

30838456 SNRIs (duloxetine)
- Says duloxetine affects serotonin and norepinephrine "in a pretty even way" while giving a 10:1 ratio.
- Says serotonin syndrome happens with "certain other medications" instead of naming MAOIs.

16445274 Hyperkalemia
- Leaves out that IV calcium doesn't lower potassium, and the beta-agonist option.
- Leaves out the common triggers (new medications, illness, dehydration) and the diabetes-related cause.

9803200 Diabetes diagnosis
- Says to start screening at 45 "if you're at risk", but 45 is for people without risk factors; people with risk factors start earlier.
- Lists "high cholesterol" as a risk factor where the abstract says high triglycerides, and leaves out impaired glucose.
- Makes diagnosis sound like one fasting blood sugar check, leaving out that it takes two abnormal tests.

21636864 Popliteal (Baker's) cyst
- Treats the study as the reader's own surgery ("your doctor reviewed your case") and never says it was 11 patients.
- Says "most of the patients" improved, but all of them did.
- Says they "couldn't fix a valve in your knee that wasn't completely open", but they failed to find the valve opening in one patient.

16127951 Hyperthyroidism
- Says the TSH test measures the thyroid hormone your body makes, but TSH is a different hormone that tells the thyroid what to do.
- Leaves out radioactive iodine, which the abstract says is the treatment of choice in the US.
- Leaves out toxic adenoma as a cause, and when surgery is used (other treatments fail or a goiter presses on the neck).

15156437 Strep throat antibiotics
- Talks as if the study were the reader's own treatment ("we compared two antibiotics... treating your tonsillitis").
- Leaves out the numbers (9 trials, 2,113 patients, 2.29 odds of clinical cure) and that it's about strep throat in adults.
- Adds that penicillin "often leads to more issues".

1000139 Clozapine vs. perphenazine
- Says clozapine was "better at affecting certain parts of the brain" and perphenazine "more effective in others", instead of which signals each one blocks.
- Leaves out the prediction that perphenazine is the stronger antipsychotic and weaker sedative.

7698623 Swallowed foreign bodies
- Says "you have a lot of small objects stuck in your stomach and throat", but it's a report on 242 different cases.
- Leaves out the success rate (98.8% removed with an endoscope) and that coins and batteries usually pass once they reach the stomach.

3068693 Finger pain
- Leaves out all the nerve and whole-body causes: carpal tunnel syndrome, rheumatoid arthritis, vasculitis, HPOA, and reflex sympathetic dystrophy.
- Leaves out the specific infections (cellulitis, paronychia, felon).

Patterns

gemma says fewer wrong things than qwen, and it gets the parts qwen flipped right: NA is norepinephrine, and insulin lowers potassium. But it leaves out much more, like radioactive iodine for hyperthyroidism, most causes of finger pain, and almost every number in the strep study. It also has the same prompt problem as qwen and treats research abstracts as the reader's own case. So it's a tradeoff: gemma is faster and makes fewer wrong statements, but drops more information.